# Домашнее задание №15: Оптимизация гиперпараметров модели

### 1. Опишите, для чего используют оптимизацию гиперпараметров?

Гиперпараметры (глубина дерева, learning rate, сила регуляризации) выставляются вручную до обучения, и от них напрямую зависит качество модели, но лучшие значения заранее неизвестны. Оптимизация — автоматический поиск комбинации, дающей лучшую метрику на валидации. Вручную это гадание: комбинаций сотни, перебирать глазами долго и почти наверняка мимо оптимума.

### 2. Какой риск возможен

Главный — переобучение на валидации: если прогнать сотни комбинаций и выбрать лучшую по одной и той же валидационной выборке, к ней подстраиваются и сами гиперпараметры. Метрика на валидации льстиво завышена, на новых данных — хуже. Лечится нетронутым тестом или nested cross-validation (NCV). Второй риск — ресурсы: каждая попытка это полное обучение модели, жадный поиск может считаться сутками ради копеечного прироста.

NCV (вложенная кросс-валидацию) - Это специальный метод проверки качества в машинном обучении. Он нужен тогда, когда мы не просто обучаем модель, но и подбираем для нее лучшие настройки (гиперпараметры). Обычная кросс-валидация в таком случае начинает «обманывать» нас, выдавая слишком оптимистичные результаты, а вложенная — спасает от этого переобучения.

### 3. Перечислите методы оптимизации.

- Grid Search — полный перебор заданной сетки, надёжно, но сетка растёт взрывно (GridSearchCV).
- Random Search — случайные комбинации из тех же диапазонов, дешевле, часто не хуже (RandomizedSearchCV).
- Байесовская оптимизация / TPE — каждая следующая попытка выбирается с учётом прошлых, целится в перспективные области (Optuna, Hyperopt).
- Эволюционные / генетические алгоритмы — комбинации скрещиваются, выживают сильнейшие.
- Методы с ранним отсевом — провальные попытки обрывают досрочно, не тратя время на дообучение (Hyperband).

### 4. Практическая часть

In [7]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

data = pd.read_csv('./data/SouthGermanCredit_encoded.csv')
raw_X = data.iloc[:, :-1]  # все столбцы кроме последнего
y = data.iloc[:, -1]   # целевая переменная "Кредитный риск" (0:плохой, 1:хороший)
scaler = MinMaxScaler()
X = scaler.fit_transform(raw_X)

In [8]:
# %pip install optuna

In [9]:
import optuna
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score

def objective(trial):
    model = Lasso(alpha=trial.suggest_float('alpha', 0.00001, 10000, log=True))
    return cross_val_score(model, X, y, cv=2).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=500)

model_l2 = Lasso(**study.best_params).fit(X, y)
print(study.best_params, model_l2.score(X, y))

[I 2026-09-16 20:47:52,755] A new study created in memory with name: no-name-1966af7c-bd4d-4b08-8d39-6d531eb879a2
[I 2026-09-16 20:47:52,770] Trial 0 finished with value: -5.2228166211841 and parameters: {'alpha': 203.52274062693309}. Best is trial 0 with value: -5.2228166211841.
[I 2026-09-16 20:47:52,780] Trial 1 finished with value: -5.2228166211841 and parameters: {'alpha': 0.19269588146846792}. Best is trial 0 with value: -5.2228166211841.
[I 2026-09-16 20:47:52,796] Trial 2 finished with value: -5.2228166211841 and parameters: {'alpha': 61.62356126308384}. Best is trial 0 with value: -5.2228166211841.
[I 2026-09-16 20:47:52,807] Trial 3 finished with value: -5.2228166211841 and parameters: {'alpha': 0.12907242144464517}. Best is trial 0 with value: -5.2228166211841.
[I 2026-09-16 20:47:52,827] Trial 4 finished with value: -3.8455497503125255 and parameters: {'alpha': 0.00018033042006607476}. Best is trial 4 with value: -3.8455497503125255.
[I 2026-09-16 20:47:52,842] Trial 5 fini

{'alpha': 0.00036898660198362595} 0.28376755483027427


Попробовал с Lasso, Ridge, ElasticNet

Но какие бы альфы не вставлял, LinearRegression всех уделала на микропункты

Хотя на самом деле это ни о чём не говорит 🤷‍♂️